# 1. Data selection

**Component 3 – Temporal Product Quality & Emerging Issue Detection (PP1)**

Choose the smartphone and laptop subset that the rest of the component is built on, using the
Amazon Reviews 2023 dataset (McAuley Lab).

Selection rules (from the PP1 plan):

1. Smartphones come from *Cell Phones & Accessories*, laptops from *Electronics*.
2. A product is a **`parent_asin`** group, so colour/storage variants are not mixed up.
3. Keep products with **>= 500 reviews** and **>= 2 years** of review history.
4. Target **30-50 products** in total (about 25 per type, at most 5 per brand).

Steps:

| Step | What happens | Output |
| --- | --- | --- |
| 1 | Scan the metadata files, keep real phones / laptops with enough ratings | `data/raw/meta_candidates.parquet` |
| 2 | Scan the review files, keep only reviews of those candidates | `data/raw/reviews_candidates_*.jsonl` |
| 3 | Count reviews and history per `parent_asin`, apply the rules, pick the subset | in memory |
| 4 | Save the selection and its reviews, draw the summary figure | `data/processed/products_selected.csv`, `data/raw/reviews_selected.parquet`, `outputs/figures/00_selected_products.png` |

The review files are large (Electronics is tens of GB), so steps 1-2 stream the files and keep only
what is needed. They are cached: re-running the notebook skips any step whose output already exists.

## Configuration

In [2]:
import json
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# Project paths (works whether the notebook is started from notebooks/ or the project root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
for folder in (RAW, PROCESSED, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

# Amazon Reviews 2023 raw files. A file already present in data/raw (same file name) is used
# instead of downloading, so you can also fetch the files by hand.
HF_RAW = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"

# dataset category -> product type we take from it
CATEGORIES = {
    "Cell_Phones_and_Accessories": "smartphone",
    "Electronics": "laptop",
}

# Selection rules
MIN_REVIEWS = 500            # reviews per parent_asin
MIN_HISTORY_YEARS = 2.0      # first to last review
TARGET_PER_TYPE = 25         # 2 types x 25 = 50 (upper end of the 30-50 target)
MAX_PER_BRAND = 5            # keeps the subset from being one brand
CANDIDATE_POOL = 150         # per type, top by rating_number; bounds the review scan
SPARSE_QUARTER_THRESHOLD = 5 # a quarter with fewer reviews than this counts as sparse
EXCLUDE_REFURBISHED = True   # renewed/refurbished listings mix seller condition into the reviews
MANUAL_EXCLUDE = set()       # parent_asin values to drop after eyeballing the result

## Helpers

* `iter_lines` streams a `.jsonl` file from disk or over HTTP (with resume if the connection drops).
* `is_target_product` separates real phones/laptops from the accessories that share the same
  Amazon categories (cases, chargers, bags, ...). The `categories` field is often empty in the 2023
  metadata, so the product title does most of the work. The selected titles are printed at the end
  so this can be checked by eye.

In [3]:
def source_for(kind, category):
    """Local file in data/raw if present, otherwise the Hugging Face URL."""
    if kind == "meta":
        name, url = f"meta_{category}.jsonl", f"{HF_RAW}/meta_categories/meta_{category}.jsonl"
    else:
        name, url = f"{category}.jsonl", f"{HF_RAW}/review_categories/{category}.jsonl"
    local = RAW / name
    return local if local.exists() else url


def iter_lines(source, max_retries=10):
    """Yield the lines (bytes, no newline) of a local or remote .jsonl file."""
    if isinstance(source, Path):
        size = source.stat().st_size
        with open(source, "rb") as f, tqdm(total=size, unit="B", unit_scale=True, desc=source.name) as bar:
            for line in f:
                bar.update(len(line))
                yield line.rstrip(b"\r\n")
        return

    offset, remainder, retries, total = 0, b"", 0, None
    bar = tqdm(unit="B", unit_scale=True, desc=source.rsplit("/", 1)[-1])
    try:
        while True:
            try:
                headers = {"Range": f"bytes={offset}-"} if offset else {}
                with requests.get(source, headers=headers, stream=True, timeout=60) as resp:
                    resp.raise_for_status()
                    if total is None:
                        total = int(resp.headers.get("Content-Length", 0)) + offset
                        bar.total = total or None
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        offset += len(chunk)
                        bar.update(len(chunk))
                        *lines, remainder = (remainder + chunk).split(b"\n")
                        for line in lines:
                            if line:
                                yield line
                if remainder.strip():
                    yield remainder
                return
            except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError):
                retries += 1
                if retries > max_retries:
                    raise
                time.sleep(min(30, 2 ** retries))  # then resume from `offset`
    finally:
        bar.close()


ACCESSORY_WORDS = {
    "smartphone": r"case|cover|protector|tempered|charger|charging|cable|adapter|holder|mount|stand|"
                  r"strap|armband|lanyard|earbuds?|headphones?|headset|stylus|wallet|pouch|sleeve|"
                  r"bumper|skin|decal|camera lens|lens kit|power bank|replacement|repair|dock|car kit|film|"
                  r"battery case|sim (card )?tray|ejector|selfie|tripod|gimbal|ring light|"
                  r"cleaning|cleaner|screen guard",
    "laptop": r"bag|backpack|sleeve|case|cover|protector|charger|adapter|power supply|"
              r"stand|riser|cooling pad|cooler|skin|decal|sticker|docking station|dock|usb hub|cable|"
              r"replacement|laptop battery|stylus|cleaning|cleaner|desk|cart",
}
INCLUDE_TITLE = {
    "smartphone": r"smartphone|smart phone|cell ?phone|unlocked phone|android phone|iphone|galaxy|"
                  r"pixel|oneplus|moto |motorola|nokia|xiaomi|redmi|oppo|realme|huawei|"
                  r"sony xperia|blu |tcl|zte|nothing phone",
    "laptop": r"laptop|notebook|chromebook|macbook|ultrabook|thinkpad|ideapad|inspiron|zenbook|"
              r"vivobook|pavilion|latitude|surface laptop",
}
INCLUDE_CATEGORY = {
    "smartphone": {"cell phones", "unlocked cell phones", "smartphones"},
    "laptop": {"laptops", "traditional laptops", "2 in 1 laptops", "chromebooks"},
}
REFURBISHED = re.compile(r"renewed|refurbished|pre-?owned|open box|reconditioned", re.I)


def is_target_product(title, categories, product_type):
    t = title.lower()
    if re.search(rf"\b(?:{ACCESSORY_WORDS[product_type]})\b", t):
        return False
    cats = {c.strip().lower() for c in categories}
    return bool(cats & INCLUDE_CATEGORY[product_type]) or bool(re.search(INCLUDE_TITLE[product_type], t))

## Step 1 · Metadata → candidate products

Scan each metadata file and keep products that look like a phone/laptop and have at least
`MIN_REVIEWS` ratings (`rating_number` is an upper bound for the number of reviews, so this is a safe
first filter). Per type, only the `CANDIDATE_POOL` products with the most ratings go on to the
review scan.

In [4]:
RATING_RE = re.compile(rb'"rating_number": ?(\d+)')


def scan_metadata(category, product_type):
    rows = []
    for line in iter_lines(source_for("meta", category)):
        m = RATING_RE.search(line)              # cheap pre-check before parsing the whole record
        if not m or int(m.group(1)) < MIN_REVIEWS:
            continue
        rec = json.loads(line)
        title = rec.get("title") or ""
        categories = rec.get("categories") or []
        if not title or not is_target_product(title, categories, product_type):
            continue
        refurbished = bool(REFURBISHED.search(title))
        if EXCLUDE_REFURBISHED and refurbished:
            continue
        rows.append({
            "parent_asin": rec["parent_asin"],
            "product_type": product_type,
            "title": title,
            "store": (rec.get("store") or "").strip(),
            "price": pd.to_numeric(rec.get("price"), errors="coerce"),
            "average_rating": rec.get("average_rating"),
            "rating_number": rec.get("rating_number"),
            "is_refurbished": refurbished,
            "category_path": " > ".join(categories),
        })
    return pd.DataFrame(rows)


META_CACHE = RAW / "meta_candidates.parquet"
if META_CACHE.exists():
    meta = pd.read_parquet(META_CACHE)
else:
    parts = []
    for category, product_type in CATEGORIES.items():
        df = scan_metadata(category, product_type)
        df = df.sort_values("rating_number", ascending=False).head(CANDIDATE_POOL)
        parts.append(df)
    meta = pd.concat(parts, ignore_index=True)
    meta.to_parquet(META_CACHE, index=False)

print(meta.groupby("product_type").size().rename("candidates").to_string())
meta.head()

meta_Cell_Phones_and_Accessories.jsonl: 0.00B [00:00, ?B/s]

KeyboardInterrupt: 

## Step 2 · Reviews of the candidates

Stream each review file and copy only the lines whose `parent_asin` is a candidate. Lines are matched
with a regular expression instead of being parsed as JSON, which keeps the scan of the very large
files as cheap as possible.

In [ ]:
PARENT_RE = re.compile(rb'"parent_asin": ?"([A-Za-z0-9]+)"')


def scan_reviews(category, wanted, out_path):
    wanted = {w.encode() for w in wanted}
    tmp = out_path.with_suffix(".part")
    kept = 0
    with open(tmp, "wb") as out:
        for line in iter_lines(source_for("review", category)):
            m = PARENT_RE.search(line)
            if m and m.group(1) in wanted:
                out.write(line + b"\n")
                kept += 1
    tmp.replace(out_path)   # only appears once the scan finished
    return kept


review_files = []
for category, product_type in CATEGORIES.items():
    out_path = RAW / f"reviews_candidates_{product_type}.jsonl"
    review_files.append(out_path)
    if out_path.exists():
        print(f"{out_path.name}: cached")
        continue
    wanted = meta.loc[meta["product_type"] == product_type, "parent_asin"]
    kept = scan_reviews(category, wanted, out_path)
    print(f"{out_path.name}: kept {kept:,} reviews of {len(wanted)} candidate products")

## Step 3 · Per-product statistics and selection

Per `parent_asin`: number of reviews, number of colour/size variants (`asin`), first and last review,
history length, and how many quarters in that span are sparse. Review timestamps are in milliseconds.
Raw counts are used here; empty and duplicate reviews are removed in the cleaning step.

In [ ]:
def load_review_index(paths):
    parent, asin, ts = [], [], []
    for path in paths:
        with open(path, "rb") as f:
            for line in f:
                r = json.loads(line)
                parent.append(r["parent_asin"])
                asin.append(r["asin"])
                ts.append(r["timestamp"])
    df = pd.DataFrame({"parent_asin": parent, "asin": asin, "timestamp": ts})
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["quarter"] = df["date"].dt.to_period("Q")
    return df


def product_stats(index):
    rows = []
    for parent_asin, g in index.groupby("parent_asin"):
        per_quarter = g.groupby("quarter").size()
        span = pd.period_range(per_quarter.index.min(), per_quarter.index.max(), freq="Q")
        per_quarter = per_quarter.reindex(span, fill_value=0)   # quarters with no reviews count too
        rows.append({
            "parent_asin": parent_asin,
            "n_reviews": len(g),
            "n_variants": g["asin"].nunique(),
            "first_review": g["date"].min().date(),
            "last_review": g["date"].max().date(),
            "history_years": round((g["date"].max() - g["date"].min()).days / 365.25, 2),
            "n_quarters_span": len(span),
            "n_active_quarters": int((per_quarter > 0).sum()),
            "sparse_quarter_share": round(float((per_quarter < SPARSE_QUARTER_THRESHOLD).mean()), 3),
        })
    return pd.DataFrame(rows)


review_index = load_review_index(review_files)
stats = product_stats(review_index)
candidates = meta.merge(stats, on="parent_asin", how="inner")
print(f"{len(review_index):,} reviews over {len(candidates)} candidate products")

candidates["eligible"] = (
    (candidates["n_reviews"] >= MIN_REVIEWS)
    & (candidates["history_years"] >= MIN_HISTORY_YEARS)
    & ~candidates["parent_asin"].isin(MANUAL_EXCLUDE)
)
print(candidates.groupby("product_type")["eligible"].agg(candidates="size", eligible="sum").to_string())

In [ ]:
selected_parts = []
for product_type, g in candidates[candidates["eligible"]].groupby("product_type"):
    g = g.sort_values("n_reviews", ascending=False)
    brand = g["store"].replace("", "unknown").str.lower()
    g = g[g.groupby(brand).cumcount() < MAX_PER_BRAND]       # brand cap, keeps the best-reviewed ones
    selected_parts.append(g.head(TARGET_PER_TYPE))

selected = pd.concat(selected_parts, ignore_index=True)

columns = [
    "parent_asin", "product_type", "title", "store", "price", "average_rating", "rating_number",
    "n_reviews", "n_variants", "first_review", "last_review", "history_years",
    "n_quarters_span", "n_active_quarters", "sparse_quarter_share",
]
selected = selected[columns].sort_values(["product_type", "n_reviews"], ascending=[True, False]).reset_index(drop=True)

print(f"Selected {len(selected)} products")
if not 30 <= len(selected) <= 50:
    print("WARNING: outside the 30-50 target. Relax MIN_REVIEWS / MIN_HISTORY_YEARS / MAX_PER_BRAND "
          "or raise CANDIDATE_POOL, then delete data/raw/meta_candidates.parquet to rescan.")
selected[["product_type", "title", "store", "n_reviews", "history_years", "sparse_quarter_share"]]

### Sanity checks

Read the titles above once: if an accessory slipped through, add its `parent_asin` to `MANUAL_EXCLUDE`
(or extend the keyword lists) and re-run from Step 3.

In [ ]:
assert selected["parent_asin"].is_unique
assert (selected["n_reviews"] >= MIN_REVIEWS).all()
assert (selected["history_years"] >= MIN_HISTORY_YEARS).all()
assert selected.groupby("product_type").size().le(TARGET_PER_TYPE).all()

summary = selected.groupby("product_type").agg(
    products=("parent_asin", "size"),
    reviews=("n_reviews", "sum"),
    median_reviews=("n_reviews", "median"),
    median_history_years=("history_years", "median"),
    median_sparse_share=("sparse_quarter_share", "median"),
)
summary

## Step 4 · Save the selection

* `data/processed/products_selected.csv` – the selected products and their statistics.
* `data/raw/reviews_selected.parquet` – all reviews of the selected products, input for the cleaning notebook.

In [ ]:
selected.to_csv(PROCESSED / "products_selected.csv", index=False)

keep = set(selected["parent_asin"])
fields = ["parent_asin", "asin", "user_id", "rating", "title", "text",
          "timestamp", "helpful_vote", "verified_purchase"]
rows = []
for path in review_files:
    with open(path, "rb") as f:
        for line in f:
            r = json.loads(line)
            if r["parent_asin"] in keep:
                rows.append({k: r.get(k) for k in fields})
reviews_selected = pd.DataFrame(rows)
reviews_selected.to_parquet(RAW / "reviews_selected.parquet", index=False)

assert len(reviews_selected) == selected["n_reviews"].sum()
print(f"products_selected.csv: {len(selected)} products")
print(f"reviews_selected.parquet: {len(reviews_selected):,} reviews")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, 0.22 * len(selected))),
                               gridspec_kw={"width_ratios": [3, 2]})
colors = {"smartphone": "#1f77b4", "laptop": "#ff7f0e"}

ordered = selected.sort_values("n_reviews")
ax1.barh(ordered["title"].str.slice(0, 45), ordered["n_reviews"],
         color=ordered["product_type"].map(colors))
ax1.set_xlabel("Reviews")
ax1.set_title("Reviews per selected product")
ax1.tick_params(axis="y", labelsize=7)

for product_type, g in selected.groupby("product_type"):
    ax2.scatter(g["history_years"], g["sparse_quarter_share"], label=product_type,
                color=colors[product_type], alpha=0.8)
ax2.set_xlabel("Review history (years)")
ax2.set_ylabel(f"Share of quarters with < {SPARSE_QUARTER_THRESHOLD} reviews")
ax2.set_title("History length vs sparsity")
ax2.legend()

fig.tight_layout()
fig.savefig(FIGURES / "1_selected_products.png", dpi=150)
plt.show()